In [1]:
import pandas as pd
from pathlib import Path

In [2]:
db_path = Path(r"D:\BDDPlabacomCoordinador")
to_save_path = Path(r"D:\ProyectoAnalisisElectrico\MedidasValorizadas")

In [4]:
dst_changes = {
    "2604": {"day": 4, "jump": -1, "instant": "24:00"},
    "2509": {"day": 6, "jump": 1,  "instant": "24:00"}
}

In [5]:
measurements_list = [
    "Medidas_Valorizadas_15min_Norte Distribución",
    "Medidas_Valorizadas_15min_Norte",
    "Medidas_Valorizadas_15min_Sur Distribución",
    "Medidas_Valorizadas_15min_Sur"
]

In [ ]:
# Define las reglas de agregación por variable (ej. costo marginal promedio, valorización total)
agg_rules = {
    'medida_3': ['sum', 'min'], 
    'CMg[CLP/KWh]': 'mean', 
    'valorizado_CLP': 'sum',
    'Fecha_Medicion': 'last',
    'CMg[USD/MWh]': 'mean'
}

# Define las dimensiones espaciales, temporales y de los agentes para agrupar los datos
group_columns = [
    'Hora', 'clave', 'nombre_barra', 'tension', 
    'Zona', 'Razon_Social', 'RUT', 'Nombre_Corto', 'tipo'
]

In [ ]:
for folder_date in db_path.iterdir():
    if not folder_date.is_dir():
        continue
        
    origin, date, version = folder_date.name.split("_")
    
    # Filtra carpetas anteriores a la fecha de corte (ej. 2505)
    if int(date) < 2505: 
        continue

    dst_info = dst_changes.get(date)
    to_save_folder = to_save_path / f"{date}"
    
    # Evita reprocesar fechas ya calculadas
    if to_save_folder.is_dir():
        print(f"Data for {date} already processed. Skipping...")
        continue 
        
    to_save_folder.mkdir(parents=True, exist_ok=True)
    measurements_folder = folder_date / "02 Medidas por tipo"

    dfs = []
    dfs_bad = []

    print(f"Starting {date}...")

    for measurement_name in measurements_list:
        csv_path = measurements_folder / measurement_name / "{}.csv".format(measurement_name)

        if not csv_path.exists():
            continue

        df = pd.read_csv(csv_path, sep=";", dtype={"clave": str})  

        # Filtra por instalaciones de tipo 'L' o 'L_D' y mantiene solo RUTs corporativos (>= 50 millones)
        df = df[(df["tipo"] == "L") | (df["tipo"] == "L_D")]
        df = df[pd.to_numeric(df["RUT"].str.split("-").str[0].str.replace(".", "", regex=False), errors="coerce") >= 50000000] 
        
        # Convierte fechas y crea un índice 'Hora' continuo a lo largo del mes
        df["Fecha_Medicion"] = pd.to_datetime(df["Fecha_Medicion"], format="%Y-%m-%d %H:%M:%S")
        df["Hora"] = (df["Fecha_Medicion"].dt.day-1)*24 + df["Fecha_Medicion"].dt.hour 

        groups = df.groupby(by=group_columns, sort=False) 
        groups_with_size = groups.size()
        
        # Identifica la hora duplicada si hubo cambio a horario de invierno (salto = -1)
        allowed_repeated_hour = -1 
        if dst_info and dst_info["jump"] == -1:
            allowed_repeated_hour = (dst_info["day"] - 1) * 24 + 23
            
        group_hours = groups_with_size.index.get_level_values('Hora')
        
        # Valida grupos: lo normal son 4 registros/hora (datos 15-min). En invierno se permiten 8 en la hora repetida.
        mask_4 = (groups_with_size == 4)
        valid_mask_8 = (groups_with_size == 8) & (group_hours == allowed_repeated_hour)
        
        good_groups_mask = mask_4 | valid_mask_8
        df_aggregated = groups.agg(agg_rules)
        
        # Corrige los datos sumados en la hora repetida de invierno (se divide a la mitad)
        if valid_mask_8.any():
            df_aggregated.loc[valid_mask_8, ('medida_3', 'sum')] /= 2 
            df_aggregated.loc[valid_mask_8, ('valorizado_CLP', 'sum')] /= 2

        # Aplana los nombres de las columnas resultantes (ej. de ('medida_3', 'sum') a 'medida_3_sum')
        df_valid = df_aggregated[good_groups_mask].reset_index()
        df_valid.columns = [f"{col[0]}_{col[1]}" if isinstance(col, tuple) and col[1] else col[0] for col in df_valid.columns]
        
        # Interpola los datos de la hora faltante si hubo cambio a horario de verano (salto = 1)
        if dst_info and dst_info["jump"] == 1:
            missing_hour = dst_info["day"] * 24
            
            if missing_hour not in df_valid['Hora'].values:
                prev_hour = missing_hour - 1
                next_hour = missing_hour + 1
                
                df_prev = df_valid[df_valid['Hora'] == prev_hour].set_index('clave')
                df_next = df_valid[df_valid['Hora'] == next_hour].set_index('clave')
                
                common_keys = df_prev.index.intersection(df_next.index)
                
                if not common_keys.empty:
                    df_missing = pd.DataFrame(index=common_keys)
                    df_missing['Hora'] = missing_hour
                    
                    num_cols = [c for c in df_valid.columns if c.endswith('_sum') or c.endswith('_mean') or c.endswith('_min')]
                    cat_cols = [c for c in df_valid.columns if c not in num_cols and c != 'Hora' and c != 'clave']
                    
                    # Mantiene las variables categóricas de la hora anterior y promedia las numéricas
                    for c in cat_cols:
                        if c == 'Fecha_Medicion_last':
                            df_missing[c] = df_prev.loc[common_keys, c] + pd.Timedelta(hours=1)
                        else:
                            df_missing[c] = df_prev.loc[common_keys, c]
                        
                    for c in num_cols:
                        df_missing[c] = (df_prev.loc[common_keys, c] + df_next.loc[common_keys, c]) / 2.0
                        
                    df_missing = df_missing.reset_index(names='clave')
                    
                    df_valid = pd.concat([df_valid, df_missing], ignore_index=True)
                    df_valid = df_valid.sort_values(by=['clave', 'Hora']).reset_index(drop=True)

        dfs.append(df_valid)
        
        # Identifica y guarda los grupos con cantidad anómala de registros (distinto de 4 o del caso válido de 8)
        bad_groups_mask = ~good_groups_mask
        if bad_groups_mask.any(): 
            print(f"Issues detected in: {measurement_name}")
            
            def is_invalid_group(x):
                sz = len(x)
                if sz == 4:
                    return False
                if sz == 8 and dst_info and dst_info["jump"] == -1:
                    if x['Hora'].iloc[0] == allowed_repeated_hour:
                        return False
                return True

            bad_df = groups.filter(is_invalid_group)
            dfs_bad.append(bad_df)
            
    # Exporta los datos consolidados válidos (Parquet) y el reporte de errores (CSV)
    if dfs:
        final_valid_df = pd.concat(dfs, ignore_index=True)
        final_valid_df.to_parquet(to_save_folder / f"{date}_medidas_horarias.parquet", engine="pyarrow", compression="snappy")
        
    if dfs_bad:
        final_bad_df = pd.concat(dfs_bad, ignore_index=True)
        final_bad_df.to_csv(to_save_folder / f"{date}_auditoria_errores_15min.csv", sep=";", index=False, encoding="utf-8")

print("\nProcess Finished.")

Starting 2505...
Index(['nombre_barra', 'tension', 'clave', 'nro_lt', 'Cuarto de Hora',
       'Fecha_Medicion', 'descripcion', 'error', 'ID_Contrato', 'Punto_Retiro',
       'Leido/Calculado', 'Modo_Calculo', 'medida_1', 'medida_2', 'medida_2a',
       'medida_3', 'Zona', 'Precio', 'Razon_Social', 'RUT', 'Nombre_Corto',
       'tipo', 'CMg[CLP/KWh]', 'valorizado_CLP'],
      dtype='str')
Starting 2506...


KeyboardInterrupt: 

In [5]:
df = pd.read_parquet(to_save_path / "2505" / "2505_medidas_horarias.parquet")
df.head(30)

,Hora,clave,nombre_barra,tension,Zona,Razon_Social,RUT,Nombre_Corto,tipo,medida_3_sum,medida_3_min,CMg[CLP/KWh]_mean,valorizado_CLP_sum,Fecha_Medicion_last
0,0,36142444,A.HOSPICIO,13,Norte Distribución,AES Andes S.A.,94.272.000-9,AES_GENER,L_D,-33.18,-8.68,70.097555,-2325.86282,2025-05-01 00:45:00
1,1,36142444,A.HOSPICIO,13,Norte Distribución,AES Andes S.A.,94.272.000-9,AES_GENER,L_D,-33.60,-8.82,68.540260,-2304.28312,2025-05-01 01:45:00
2,2,36142444,A.HOSPICIO,13,Norte Distribución,AES Andes S.A.,94.272.000-9,AES_GENER,L_D,-34.58,-8.96,63.913277,-2210.00833,2025-05-01 02:45:00
3,3,36142444,A.HOSPICIO,13,Norte Distribución,AES Andes S.A.,94.272.000-9,AES_GENER,L_D,-35.00,-9.38,66.770067,-2334.46393,2025-05-01 03:45:00
4,4,36142444,A.HOSPICIO,13,Norte Distribución,AES Andes S.A.,94.272.000-9,AES_GENER,L_D,-33.18,-8.82,69.821867,-2316.51870,2025-05-01 04:45:00
5,5,36142444,A.HOSPICIO,13,Norte Distribución,AES Andes S.A.,94.272.000-9,AES_GENER,L_D,-32.20,-8.40,70.002730,-2254.19491,2025-05-01 05:45:00
6,6,36142444,A.HOSPICIO,13,Norte Distribución,AES Andes S.A.,94.272.000-9,AES_GENER,L_D,-31.22,-8.12,66.252512,-2068.86091,2025-05-01 06:45:00
7,7,36142444,A.HOSPICIO,13,Norte Distribución,AES Andes S.A.,94.272.000-9,AES_GENER,L_D,-33.46,-8.82,57.808303,-1949.55975,2025-05-01 07:45:00
8,8,36142444,A.HOSPICIO,13,Norte Distribución,AES Andes S.A.,94.272.000-9,AES_GENER,L_D,-34.86,-9.38,11.079280,-387.29844,2025-05-01 08:45:00
9,9,36142444,A.HOSPICIO,13,Norte Distribución,AES Andes S.A.,94.272.000-9,AES_GENER,L_D,-38.50,-9.80,2.621460,-99.82520,2025-05-01 09:45:00


In [4]:
df.columns

Index(['Hora', 'clave', 'nombre_barra', 'tension', 'Zona', 'Razon_Social',
       'RUT', 'Nombre_Corto', 'tipo', 'medida_3_sum', 'medida_3_min',
       'CMg[CLP/KWh]_mean', 'valorizado_CLP_sum', 'Fecha_Medicion_last'],
      dtype='str')